## Import

In [ ]:
import pandas as pd
import numpy as np
import os
import random

from sklearn.model_selection import train_test_split
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

## Setting

In [ ]:
CFG = {
    'BATCH_SIZE': 4096,
    'EPOCHS': 10,
    'LEARNING_RATE': 1e-3,
    'SEED' : 42
}
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def seed_everything(seed):
    # 파이썬 내장 random 모듈의 시드 고정 (난수 생성 일관성 확보)
    random.seed(seed)
    
    # 파이썬 해시(seed) 값 고정 (실행마다 문자열 해싱 결과가 달라지는 문제 방지)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 넘파이 난수 시드 고정 (배열 연산이나 샘플링의 일관성 확보)
    np.random.seed(seed)
    
    # PyTorch CPU 연산의 난수 시드 고정
    torch.manual_seed(seed)
    
    # PyTorch GPU 연산(CUDA)의 난수 시드 고정
    torch.cuda.manual_seed(seed)
    
    # cuDNN(딥러닝 연산 가속 라이브러리)의 연산을 매번 동일하게 수행하도록 설정
    torch.backends.cudnn.deterministic = True  # 연산 결과를 결정론적으로 고정
    torch.backends.cudnn.benchmark = False     # 하드웨어 최적화 탐색 비활성화 (재현성 우선)

# 설정된 CFG['SEED'] 값을 사용해 전체 환경의 난수 시드를 고정
seed_everything(CFG['SEED'])


## Data Load

In [ ]:
# ==============================
# 데이터 로드
# ==============================

# 학습 데이터 불러오기
# - 파일 형식: Parquet (대용량 데이터 처리를 위한 효율적 컬럼 기반 저장 형식)
# - engine="pyarrow": 파이썬에서 Parquet 파일을 읽을 때 pyarrow 라이브러리를 사용
all_train = pd.read_parquet("./train.parquet", engine="pyarrow")

# 테스트 데이터 불러오기
# - 마찬가지로 Parquet 형식을 pyarrow로 읽어옴
# - 불필요한 식별자(ID) 컬럼은 drop하여 제거 (모델 입력에는 사용하지 않음)
test = pd.read_parquet("./test.parquet", engine="pyarrow").drop(columns=['ID'])

# 불러온 데이터프레임의 크기 출력
# - shape[0]: 행(샘플 수), shape[1]: 열(피처 수)
print("Train shape:", all_train.shape)  # 학습 데이터셋 크기 확인
print("Test shape:", test.shape)        # 테스트 데이터셋 크기 확인


## Data Down-Sampling

In [ ]:
# ==============================
# 데이터 샘플링 (클래스 불균형 처리)
# ==============================

# 1) 클릭된 데이터만 추출 (clicked == 1)
# - 긍정 클래스(클릭 있음) 데이터만 필터링
clicked_1 = all_train[all_train['clicked'] == 1]

# 2) 클릭되지 않은 데이터에서 일부만 추출 (다운 샘플링)
# - 부정 클래스(클릭 없음)가 훨씬 많을 수 있으므로
# - clicked == 1 데이터 개수의 2배만큼 무작위 추출
# - random_state=42로 시드 고정 (재현성 보장)
clicked_0 = all_train[all_train['clicked'] == 0].sample(
    n=len(clicked_1)*2, random_state=42
)

# 3) 두 데이터프레임 합치기
# - 긍정 데이터(clicked_1)와 다운샘플링된 부정 데이터(clicked_0)를 합침
# - sample(frac=1)로 전체를 무작위 섞어주기 (셔플)
# - reset_index(drop=True)로 새로운 연속 인덱스 부여
train = pd.concat([clicked_1, clicked_0], axis=0) \
           .sample(frac=1, random_state=42) \
           .reset_index(drop=True)


In [ ]:
# 학습 데이터셋 크기 출력
print("Train shape:", train.shape)  
# -> 전체 학습 데이터 (행, 열) 크기를 확인

# 클릭되지 않은 데이터 개수 출력
print("Train clicked:0:", train[train['clicked'] == 0].shape)  
# -> clicked == 0 인 샘플 수를 확인

# 클릭된 데이터 개수 출력
print("Train clicked:1:", train[train['clicked'] == 1].shape)  
# -> clicked == 1 인 샘플 수를 확인

## Data Column Setting

In [ ]:
# ==============================
# 타겟 변수 및 피처 설정
# ==============================

# 1) 예측할 타겟 컬럼 정의
target_col = "clicked"   # 클릭 여부 (분류 문제에서 종속 변수)

# 2) 시퀀스 관련 컬럼 정의
seq_col = "seq"          # 순서나 세션 단위로 사용될 수 있는 컬럼

# 3) 학습에 제외할 컬럼 집합 정의
# - 타겟 컬럼 (clicked), 시퀀스 컬럼 (seq), 식별자(ID) 제외
FEATURE_EXCLUDE = {target_col, seq_col, "ID"}

# 4) 실제 학습에 사용할 피처 컬럼 목록 생성
# - 제외 컬럼을 뺀 나머지 모든 컬럼을 feature_cols로 지정
feature_cols = [c for c in train.columns if c not in FEATURE_EXCLUDE]

# 5) 주요 정보 출력
print("Num features:", len(feature_cols))  # 사용되는 피처의 개수
print("Sequence:", seq_col)                # 시퀀스 변수명
print("Target:", target_col)               # 타겟 변수명


## Define Custom Dataset

In [ ]:
# ==============================
# 커스텀 데이터셋 클래스 정의 (PyTorch Dataset)
# ==============================

class ClickDataset(Dataset):
    def __init__(self, df, feature_cols, seq_col, target_col=None, has_target=True):
        # 데이터프레임 초기화 및 인덱스 리셋
        self.df = df.reset_index(drop=True)
        
        # 학습에 사용할 피처, 시퀀스 컬럼, 타겟 컬럼 정보 저장
        self.feature_cols = feature_cols
        self.seq_col = seq_col
        self.target_col = target_col
        self.has_target = has_target

        # ------------------------------
        # 비-시퀀스 피처 처리
        # ------------------------------
        # - 지정된 feature_cols만 추출
        # - float 형식으로 변환
        # - 결측치는 0으로 채움
        # - values: numpy array로 변환
        self.X = self.df[self.feature_cols].astype(float).fillna(0).values

        # ------------------------------
        # 시퀀스 피처 처리
        # ------------------------------
        # - 시퀀스 데이터를 문자열로 보관
        # - 이후 __getitem__에서 필요 시 파싱(lazy parsing)
        self.seq_strings = self.df[self.seq_col].astype(str).values

        # ------------------------------
        # 타겟 변수 처리
        # ------------------------------
        # - has_target=True일 때만 y를 생성
        if self.has_target:
            self.y = self.df[self.target_col].astype(np.float32).values

    def __len__(self):
        # 전체 샘플 개수 반환
        return len(self.df)

    def __getitem__(self, idx):
        # ------------------------------
        # 비-시퀀스 피처 반환
        # ------------------------------
        x = torch.tensor(self.X[idx], dtype=torch.float)

        # ------------------------------
        # 시퀀스 피처 반환
        # ------------------------------
        # - 문자열을 쉼표(,) 기준으로 분리해 float 배열로 변환
        s = self.seq_strings[idx]
        if s:
            arr = np.fromstring(s, sep=",", dtype=np.float32)
        else:
            arr = np.array([], dtype=np.float32)

        # 빈 시퀀스 방어 → 최소한 하나의 값([0.0]) 유지
        if arr.size == 0:
            arr = np.array([0.0], dtype=np.float32)

        seq = torch.from_numpy(arr)  # shape: (seq_len,)

        # ------------------------------
        # 타겟 반환 (학습용 / 테스트용 구분)
        # ------------------------------
        if self.has_target:
            y = torch.tensor(self.y[idx], dtype=torch.float)
            return x, seq, y
        else:
            return x, seq


In [ ]:
# ==============================
# Collate 함수 정의 (DataLoader에서 배치 구성 시 사용)
# ==============================

def collate_fn_train(batch):
    # batch: Dataset에서 반환된 (x, seq, y) 튜플들의 리스트
    
    # 1) 배치 데이터를 각각 분리
    xs, seqs, ys = zip(*batch)
    
    # 2) 비-시퀀스 피처를 텐서로 쌓기 (batch_size, num_features)
    xs = torch.stack(xs)
    
    # 3) 타겟 값도 텐서로 쌓기 (batch_size,)
    ys = torch.stack(ys)
    
    # 4) 시퀀스 데이터는 길이가 제각각 → pad_sequence로 패딩
    # - batch_first=True: (batch_size, max_seq_len) 형태로 맞춤
    # - padding_value=0.0: 부족한 길이는 0으로 채움
    seqs_padded = nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0.0)
    
    # 5) 각 시퀀스의 원래 길이를 기록
    seq_lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    seq_lengths = torch.clamp(seq_lengths, min=1)  # 빈 시퀀스가 0이 되는 것 방지
    
    # 6) (입력 피처, 패딩된 시퀀스, 시퀀스 길이, 타겟) 반환
    return xs, seqs_padded, seq_lengths, ys


def collate_fn_infer(batch):
    # batch: Dataset에서 반환된 (x, seq) 튜플들의 리스트
    
    # 1) 비-시퀀스 피처 묶기
    xs, seqs = zip(*batch)
    xs = torch.stack(xs)
    
    # 2) 시퀀스 패딩 처리
    seqs_padded = nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0.0)
    
    # 3) 시퀀스 길이 기록
    seq_lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    seq_lengths = torch.clamp(seq_lengths, min=1)
    
    # 4) (입력 피처, 패딩된 시퀀스, 시퀀스 길이) 반환
    return xs, seqs_padded, seq_lengths


## Define Model Architecture

In [ ]:
# ==============================
# Tabular + Sequence 통합 모델
# ==============================

class TabularSeqModel(nn.Module):
    def __init__(self, d_features, lstm_hidden=32, hidden_units=[1024, 512, 256, 128], dropout=0.2):
        super().__init__()

        # ------------------------------
        # 비-시퀀스 입력 처리 (탭형 피처)
        # ------------------------------
        # - BatchNorm1d: 입력 피처를 정규화하여 학습 안정성/속도 향상
        self.bn_x = nn.BatchNorm1d(d_features)

        # ------------------------------
        # 시퀀스 입력 처리 (숫자 시퀀스 → LSTM)
        # ------------------------------
        # - input_size=1: 시퀀스는 단일 차원 값(숫자)
        # - hidden_size=lstm_hidden: 시퀀스에서 추출할 특징 차원
        # - batch_first=True: 입력 형태를 (B, L, D)로 받음
        self.lstm = nn.LSTM(input_size=1, hidden_size=lstm_hidden, batch_first=True)

        # ------------------------------
        # MLP (비-시퀀스 + 시퀀스 특징 결합 후 최종 분류기)
        # ------------------------------
        input_dim = d_features + lstm_hidden  # 탭형 피처와 LSTM 출력 결합
        layers = []
        for h in hidden_units:
            layers += [nn.Linear(input_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            input_dim = h
        layers += [nn.Linear(input_dim, 1)]   # 마지막은 이진 분류를 위한 로짓 출력
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_feats, x_seq, seq_lengths):
        # ------------------------------
        # 비-시퀀스 피처 처리
        # ------------------------------
        x = self.bn_x(x_feats)  # (B, d_features)

        # ------------------------------
        # 시퀀스 처리 (LSTM)
        # ------------------------------
        x_seq = x_seq.unsqueeze(-1)  # (B, L) → (B, L, 1)
        
        # 시퀀스 길이가 다르므로 pack으로 묶어 효율적 연산
        packed = nn.utils.rnn.pack_padded_sequence(
            x_seq, seq_lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        # LSTM 통과 → (마지막 hidden state만 사용)
        _, (h_n, _) = self.lstm(packed)
        h = h_n[-1]  # (B, lstm_hidden)

        # ------------------------------
        # 결합 + MLP 분류
        # ------------------------------
        z = torch.cat([x, h], dim=1)         # (B, d_features + lstm_hidden)
        return self.mlp(z).squeeze(1)        # (B,) 로짓 출력


## Train / Validation

In [ ]:
# ==============================
# 학습 루프 함수 정의
# ==============================

def train_model(train_df, feature_cols, seq_col, target_col,
                batch_size=512, epochs=3, lr=1e-3, device="cuda"):

    # ------------------------------
    # 1) Train / Validation 데이터 분할
    # ------------------------------
    # - 학습용(train)과 검증용(val) 데이터로 8:2 비율 분리
    # - random_state=42 → 재현성 확보
    tr_df, va_df = train_test_split(train_df, test_size=0.2, random_state=42, shuffle=True)

    # ------------------------------
    # 2) Dataset / DataLoader 정의
    # ------------------------------
    train_dataset = ClickDataset(tr_df, feature_cols, seq_col, target_col, has_target=True)
    val_dataset   = ClickDataset(va_df, feature_cols, seq_col, target_col, has_target=True)

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn_train
    )
    val_loader   = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_train
    )

    # ------------------------------
    # 3) 모델 초기화
    # ------------------------------
    d_features = len(feature_cols)  # 입력 피처 개수
    model = TabularSeqModel(
        d_features=d_features,
        lstm_hidden=64,                # LSTM hidden 크기
        hidden_units=[256,128],        # MLP hidden 레이어 크기
        dropout=0.2                    # Dropout 비율
    ).to(device)

    # 손실 함수: BCEWithLogitsLoss (sigmoid + binary cross entropy)
    criterion = nn.BCEWithLogitsLoss()
    # 최적화 알고리즘: Adam
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # ------------------------------
    # 4) 학습 루프
    # ------------------------------
    for epoch in range(1, epochs+1):
        # ----- 학습 단계 -----
        model.train()
        train_loss = 0.0
        for xs, seqs, seq_lens, ys in tqdm(train_loader, desc=f"Train Epoch {epoch}"):
            # 배치 데이터를 GPU/CPU로 이동
            xs, seqs, seq_lens, ys = xs.to(device), seqs.to(device), seq_lens.to(device), ys.to(device)
            
            optimizer.zero_grad()             # 이전 기울기 초기화
            logits = model(xs, seqs, seq_lens) # forward pass
            loss = criterion(logits, ys)      # 손실 계산
            loss.backward()                   # 역전파
            optimizer.step()                  # 가중치 업데이트
            
            train_loss += loss.item() * ys.size(0)
        train_loss /= len(train_dataset)      # epoch 평균 손실

        # ----- 검증 단계 -----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xs, seqs, seq_lens, ys in tqdm(val_loader, desc=f"Val Epoch {epoch}"):
                xs, seqs, seq_lens, ys = xs.to(device), seqs.to(device), seq_lens.to(device), ys.to(device)
                logits = model(xs, seqs, seq_lens)
                loss = criterion(logits, ys)
                val_loss += loss.item() * len(ys)
        val_loss /= len(val_dataset)

        # ----- 로그 출력 -----
        print(f"[Epoch {epoch}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # 학습 완료 후 최종 모델 반환
    return model


## Run!!

In [ ]:
# ==============================
# 모델 학습 실행
# ==============================

model = train_model(
    train_df=train,                 # 학습 데이터프레임 (clicked 1:2 비율로 샘플링된 데이터)
    feature_cols=feature_cols,      # 학습에 사용할 피처 컬럼들
    seq_col=seq_col,                # 시퀀스 컬럼 이름 ("seq")
    target_col=target_col,          # 타겟 컬럼 이름 ("clicked")
    batch_size=CFG['BATCH_SIZE'],   # 배치 크기 (4096)
    epochs=CFG['EPOCHS'],           # 학습 에폭 수 (10)
    lr=CFG['LEARNING_RATE'],        # 학습률 (1e-3)
    device=device                   # 실행 장치 ("cuda" 또는 "cpu")
)


## Inference

In [ ]:
# ==============================
# 추론(Inference) 단계
# ==============================

# 1) Dataset / DataLoader 정의
# - 테스트 데이터셋 생성 (has_target=False → 라벨 없음)
test_ds = ClickDataset(test, feature_cols, seq_col, has_target=False)

# - DataLoader: 순서 유지(shuffle=False), collate_fn_infer 사용
test_ld = DataLoader(
    test_ds,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=False,
    collate_fn=collate_fn_infer
)

# 2) 모델 예측 수행
model.eval()        # 모델을 평가 모드로 전환 (Dropout, BN 등 동작 변경)
outs = []

with torch.no_grad():  # 추론 시 불필요한 gradient 계산 방지 (메모리 절약, 속도 향상)
    for xs, seqs, lens in tqdm(test_ld, desc="Inference"):
        # 배치 데이터를 실행 장치로 이동
        xs, seqs, lens = xs.to(device), seqs.to(device), lens.to(device)
        
        # 모델 예측값 계산 (로짓 → 시그모이드 확률 변환)
        logits = model(xs, seqs, lens)
        probs = torch.sigmoid(logits).cpu()  # CPU로 이동
        outs.append(probs)

# 3) 전체 배치 결과를 하나의 배열로 결합
test_preds = torch.cat(outs).numpy()  # 최종 예측 확률 (numpy array)


## Submission

In [ ]:
submit = pd.read_csv('./sample_submission.csv')
submit['clicked'] = test_preds

In [ ]:
submit.to_csv('./baseline_submit.csv', index=False)